# 🔮 ANN Churn Classification — Prediction

> **Purpose:** Load the trained ANN model and saved preprocessors to predict whether a new bank customer is likely to churn.

---

## 📋 What This Notebook Does

### 1. 📦 Load Saved Artifacts
- Imports **`model.h5`** — the trained ANN model
- Imports **`scaler.pkl`** — the fitted StandardScaler
- Imports **`label_encoder_gender.pkl`** — the fitted LabelEncoder for Gender
- Imports **`onehot_encoder_geo.pkl`** — the fitted OneHotEncoder for Geography

---

### 2. 👤 Define a Sample Customer
- Provide a **single customer's details** as input:
  - *CreditScore, Geography, Gender, Age, Tenure, Balance, NumOfProducts, HasCrCard, IsActiveMember, EstimatedSalary*

---

### 3. ⚙️ Preprocess the Input
Apply the **same transformations** used during training:
- **`OneHotEncoder`** on `Geography` → expands into 3 binary columns *(France / Germany / Spain)*
- **`LabelEncoder`** on `Gender` → converts *Male/Female* to *1/0*
- **Drop** the original `Geography` column and **concatenate** the encoded columns

---

### 4. 📏 Scale the Input
- Use the **same `StandardScaler`** fitted on training data to normalize all **12 features**
- *Ensures the model receives data in the exact same format it was trained on*

---

### 5. 🧠 Predict Churn Probability
- Pass the scaled input through the **ANN model**
- Outputs a **probability score between 0 and 1**

---

### 6. ✅ Output the Decision
| Condition | Result |
|---|---|
| Probability **> 0.5** | 🚨 Customer is **likely to churn** |
| Probability **≤ 0.5** | ✅ Customer is **likely to stay** |

---

> ⚠️ **Important:** Always use the **same encoders and scaler** saved during training.
> Never refit them on new data — that would produce a different transformation and give **wrong predictions**.

---

**Model:** Deep Learning ANN &nbsp;|&nbsp; **Framework:** TensorFlow / Keras &nbsp;|&nbsp; **Task:** Binary Classification


In [1]:
from tensorflow.keras.models import load_model
import numpy as np
import pickle
import pandas as pd

In [2]:
# Load the trained model, scaler pickle, one got from training and the other from preprocessing

model = load_model('model.h5')

# Load the encoder and scaler
with open('label_encoder_gender.pkl', 'rb') as file:
    label_encoder_gender = pickle.load(file)

with open('onehot_encoder_geo.pkl', 'rb') as file:
    onehot_encoder_geo = pickle.load(file)

with open('scaler.pkl', 'rb') as file:
    scaler = pickle.load(file)





In [3]:
# Example input data
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [4]:
# One Hot Encoding for Geography
geo_encoded = onehot_encoder_geo.transform([[input_data['Geography']]]).toarray()
geo_enoded_df = pd.DataFrame(geo_encoded, columns = onehot_encoder_geo.get_feature_names_out(['Geography']))
geo_enoded_df

c:\Users\krish\OneDrive\Desktop\AI Engineer\Deep Learning\ANN\ANN_Churn_Classification\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [5]:
input_df= pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [6]:
# Enocde categorical data

input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [7]:
# concatination one hot enoded

input_df = pd.concat([input_df.drop('Geography', axis=1), geo_enoded_df], axis=1)

In [8]:
# Scaling the data

input_scaled = scaler.transform(input_df)
input_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [9]:
# predict the output

prediction = model.predict(input_scaled)
prediction

1/1 [==============================] - 0s 93ms/step


array([[0.01803449]], dtype=float32)

In [10]:
prediction_proba = prediction[0][0]
prediction_proba

0.01803449

In [11]:
if prediction_proba > 0.5:
    print("The customer is likely to churn.")  
else:
    print("The customer is not likely to churn.")

The customer is not likely to churn.
